# Optimizer — self-recovery on a small polymer

**The honesty gate.** Invent a known θ, simulate it, treat that map as
"experimental", fit θ back, check recovery. If this fails, nothing downstream
can be trusted.

---
## Caching: read this once, then forget it

Every simulation is written to Drive and **skipped if it already exists**. That
makes long runs resumable, but it means stale results replay silently after a
code change. The dependency structure:

| artifact | tags | depends on | stale when |
|---|---|---|---|
| target + truth ensemble | `truth_*`, `target.npy` | true model, simulator physics | model/simulator changes |
| replica-noise ceiling | `ceil_*` | true model, simulator physics | model/simulator changes |
| loss scan | `scan_*` | simulator, target | model/simulator changes |
| fit iterations | `k1_it*`, `fits/*` | **optimizer code** | optimizer changes |
| behavioural check | `fitted_*` | **the fit result** | optimizer changes |

Set `RESET_LEVEL` in the setup cell and the right tier is purged automatically:

- **0** — reuse everything (normal re-run)
- **1** — redo the fit and everything after it *(use after an optimizer change)*
- **2** — redo everything *(use after a model/simulator change, or new ground truth)*

The target and ceiling cost ~15 hours; level 1 protects them.

## 0. Setup

In [ ]:
from google.colab import drive
!git clone -q https://github.com/darinddv/chromatin_potential.git
!pip install -q "openmm[cuda12]" OpenMiChroM
import sys; sys.path.insert(0, '/content/chromatin_potential/src')
drive.mount('/content/drive')

import os, glob, json, time, inspect
import numpy as np
import matplotlib.pyplot as plt

from chromatin_potential.model import Model
from chromatin_potential.simulator import (
    Simulator, BackgroundStack, SaveSpec,
    observed_over_expected, saddle_strength)
from chromatin_potential import optimizer as O

# ---- code-version check: catches "forgot to push / forgot to pull" ----
_src = inspect.getsource(O)
checks = {
    'gradient sign fixed'  : 'P_exp - P_sim' in _src,
    'best-iterate return'  : 'best_params' in _src,
    'overshoot detection'  : 'worsen_window' in _src,
}
for k, v in checks.items():
    print(f'  {"OK " if v else "MISSING"}  {k}')
assert all(checks.values()), ("optimizer.py is out of date -- push locally, "
                              "then `!git -C /content/chromatin_potential pull` "
                              "and RESTART the runtime.")
print('setup OK')

In [ ]:
RESET_LEVEL = 0      # 0 reuse all | 1 redo fit onward | 2 redo everything

DATA = '/content/drive/MyDrive/uky_cheng/tecsas_ablation'
OUT  = f'{DATA}/optimizer_selfrec'
os.makedirs(OUT, exist_ok=True)

N_BEADS = 300
names   = [f't{i:05d}' for i in range(N_BEADS)]
SEQ     = f'{OUT}/perbead_{N_BEADS}.txt'
with open(SEQ,'w') as fh:
    for i,n in enumerate(names): fh.write(f'{i+1} {n}\n')

def purge(patterns, label):
    n = 0
    for pat in patterns:
        for f in glob.glob(pat):
            os.remove(f); n += 1
    print(f'  purged {n:4d} files  ({label})')

# tier 1: anything downstream of the optimizer
TIER1 = [f'{OUT}/maps/k1_it*', f'{OUT}/traj/k1_it*', f'{OUT}/meta/k1_it*',
         f'{OUT}/maps/freeC_it*', f'{OUT}/traj/freeC_it*', f'{OUT}/meta/freeC_it*',
         f'{OUT}/fits/*.json',
         f'{OUT}/maps/fitted_rep*', f'{OUT}/traj/fitted_rep*', f'{OUT}/meta/fitted_rep*']
# tier 2: also the expensive ground-truth artifacts
TIER2 = TIER1 + [f'{OUT}/maps/truth_rep*', f'{OUT}/traj/truth_rep*', f'{OUT}/meta/truth_rep*',
                 f'{OUT}/maps/ceil_rep*',  f'{OUT}/meta/ceil_rep*',
                 f'{OUT}/maps/scan_*',     f'{OUT}/meta/scan_*',
                 f'{OUT}/target.npy']

if RESET_LEVEL >= 2:
    purge(TIER2, 'everything -- target and ceiling will be regenerated (~15 h)')
elif RESET_LEVEL == 1:
    purge(TIER1, 'fit onward -- target and ceiling preserved')
else:
    print('  reusing all cached artifacts')

print(f'\nexisting: truth={len(glob.glob(f"{OUT}/maps/truth_rep*.npy"))}'
      f'  ceil={len(glob.glob(f"{OUT}/maps/ceil_rep*.npy"))}'
      f'  target={os.path.exists(f"{OUT}/target.npy")}')

## 1. Timing check

The plan rests on a small-polymer iteration being cheap. Measure it.

In [ ]:
sim = Simulator(seq_file=SEQ, out_dir=OUT, platform='cuda',
                background=BackgroundStack())

probe = Model(O.synthetic_coordinate(N_BEADS, k=1, n_domains=12, seed=0),
              np.array([[-1.0]]), c=-0.30, names=names)
t0 = time.time()
sim.run_replica('timing', 0, model=probe, n_production=50_000,
                save_spec=SaveSpec(contact_map=True, trajectory=True,
                                   interval=1000, monitor=False),
                overwrite=True, verbose=False)
dt = time.time() - t0
print(f'50k steps: {dt/60:.2f} min  ->  4-replica iteration ~ {4*dt/60:.1f} min'
      f'  ->  60 iterations ~ {60*4*dt/3600:.1f} h')

## 2. Ground truth (k=1)

In [ ]:
C_true   = O.synthetic_coordinate(N_BEADS, k=1, n_domains=12, seed=1)
LAM_TRUE = -1.2
true_model = Model(C_true, np.array([[LAM_TRUE]]), c=-0.30, names=names)
print(f'true lambda = {LAM_TRUE}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(C_true[:,0], lw=.9); ax[0].set_title('true coordinate C'); ax[0].set_xlabel('bead')
im = ax[1].imshow(true_model.coupling_matrix(), cmap='RdBu_r')
ax[1].set_title('coupling matrix M'); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

## 3. Synthetic target  *(expensive — protected at RESET_LEVEL ≤ 1)*

Why synthetic and not a real chr10 sub-region: a sub-region's measured map
includes contacts with the rest of the chromosome and the nucleus, which an
isolated simulation cannot reproduce. Fitting it would absorb that boundary
mismatch into Λ — the fit-vs-simulate environment mismatch the retraining rule
warns about. A synthetic target is generated in exactly the environment it is
fitted in.

In [ ]:
if os.path.exists(f'{OUT}/target.npy'):
    target = np.load(f'{OUT}/target.npy')
    print('loaded cached target', target.shape)
else:
    target = O.make_synthetic_target(sim, true_model, n_replicas=20,
                                     n_production=3_000_000,
                                     cond='truth', verbose=True)
    np.save(f'{OUT}/target.npy', target)
    print('generated target', target.shape)

### 3a. Replica-noise ceiling

Correlation between two INDEPENDENT ensembles of the SAME θ. **Perfect recovery
is this number, not 1.0.** Judging against 1.0 makes a correct fit look broken.

In [ ]:
CEIL_F = f'{OUT}/ceiling.json'
if os.path.exists(CEIL_F):
    ceiling = json.load(open(CEIL_F))['ceiling']
    print(f'loaded cached ceiling = {ceiling:.5f}')
else:
    ceiling, _, _ = O.replica_noise_ceiling(sim, true_model, n_replicas=10,
                                            n_production=3_000_000,
                                            cond='ceil', verbose=False)
    json.dump({'ceiling': ceiling}, open(CEIL_F,'w'))
    print(f'ceiling = {ceiling:.5f}')

### 3b. Loss scan — where is the minimum, and how flat is it?

Independent of the optimizer: evaluate the loss at fixed λ values. Answers two
questions at once — *is the objective correct* (minimum at the true λ) and
*how well can λ be determined at all* (curvature vs replica noise).

The second is an identifiability measurement, not a diagnostic.

In [ ]:
SCAN_F = f'{OUT}/scan.json'
SCAN_LAMS = [-1.0, -1.2, -1.4, -1.6]
if os.path.exists(SCAN_F):
    scan = json.load(open(SCAN_F))
    print('loaded cached scan')
else:
    scan = []
    for i, lam in enumerate(SCAN_LAMS):
        m = Model(C_true, np.array([[lam]]), c=-0.30, names=names)
        maps = []
        for s in range(4):
            sim.run_replica(f'scan_{i}', s, model=m, n_production=170_000,
                            save_spec=SaveSpec(contact_map=True, trajectory=False,
                                               monitor=False),
                            verbose=False)
            maps.append(np.load(f'{OUT}/maps/scan_{i}_rep{s}.npy').astype(float))
        G, mask = O.map_residual(np.mean(maps, axis=0), target)
        scan.append([lam, O.loss_from_residual(G, mask)])
    json.dump(scan, open(SCAN_F,'w'))

for lam, l in scan:
    print(f'  lambda={lam:+.2f}  loss={l:.4e}')
lams = [s[0] for s in scan]; ls = [s[1] for s in scan]
print(f'\nminimum at lambda = {lams[int(np.argmin(ls))]:+.2f}  (true {LAM_TRUE:+.2f})')
spread = (max(ls[:3]) - min(ls[:3])) / min(ls[:3])
print(f'loss varies only {spread:.1%} across lambda in [{lams[0]}, {lams[2]}]')
print('-> if that is comparable to replica noise, lambda is only determined to')
print('   within that range. THAT IS THE IDENTIFIABILITY RESULT, not a failure.')

## 4. Fit

λ starts deliberately wrong (−0.3) so recovery is a real test.

The optimizer returns the **best** iterate, not the last, and stops when the loss
rises for several iterations past the minimum. Both matter on a flat, noisy
objective — without them a fit walks past the minimum and reports whatever it
happened to reach.

In [ ]:
init_model = Model(C_true.copy(), np.array([[-0.3]]), c=-0.30, names=names)
budget = O.Budget(n_steps=50_000, n_replicas=4,
                  min_steps=50_000, max_steps=400_000,
                  min_replicas=2,  max_replicas=16)

opt = O.Optimizer(sim, target, lr_lambda=0.05, fit_c=False,
                  budget=budget, out_dir=f'{OUT}/fits', tag='k1', verbose=True)
res = opt.fit(init_model, n_iter=60, patience=12)

## 5. Recovery

In [ ]:
rep = O.recovery_report(true_model, res.model)
rep['replica_noise_ceiling'] = ceiling
lam_fit = res.model.Lam[0,0]

print(f'true lambda    = {LAM_TRUE:+.4f}')
print(f'fitted lambda  = {lam_fit:+.4f}   (best iterate, iter {res.best_iter})')
print(f'last iterate   = {res.final_model.Lam[0,0]:+.4f}')
print(f'absolute error = {abs(lam_fit-LAM_TRUE):.4f}  ({abs(lam_fit-LAM_TRUE)/abs(LAM_TRUE):.1%})')
print()
print(f"M relative error = {rep['M_relative_error']:.4f}   <- PRIMARY")
print(f"M scale ratio    = {rep['M_scale_ratio']:.4f}   (1.0 = right magnitude)")
print(f"M correlation    = {rep['M_correlation']:.6f}   <- ignore: for k=1 the")
print('    off-diagonal shape is proportional to C C^T for ANY lambda, so this')
print('    reads +-1.000 regardless of magnitude.')
print(f"\nstop: {res.stop_reason}")
json.dump(rep, open(f'{OUT}/recovery_report_k1.json','w'), indent=1, default=str)

# judge against the scan, not against zero error
lams = [s[0] for s in scan]; ls = [s[1] for s in scan]
flat = abs(lams[int(np.argmin(ls))] - LAM_TRUE) + 0.2
print(f"\nVERDICT: the loss scan shows lambda is determined to roughly +-0.2.")
print(f"  error {abs(lam_fit-LAM_TRUE):.3f} -> "
      f"{'AT the noise floor: recovered as well as the data allows' if abs(lam_fit-LAM_TRUE) <= 0.2 else 'outside the flat region: investigate'}")

### 5a. Traces

In [ ]:
loss = res.trace('loss'); gn = res.trace('grad_norm')
lam  = np.array([h['Lambda'][0][0] for h in res.history])
steps = res.trace('n_steps'); reps = res.trace('n_replicas')

fig, ax = plt.subplots(1, 4, figsize=(17, 3.3))
ax[0].semilogy(loss, 'o-'); ax[0].axvline(res.best_iter, c='g', ls=':', label='best')
ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(lam, 'o-'); ax[1].axhline(LAM_TRUE, c='r', ls='--', label='true')
ax[1].axvline(res.best_iter, c='g', ls=':'); ax[1].set_title('lambda'); ax[1].legend()
ax[2].semilogy(gn, 'o-'); ax[2].set_title('|gradient|')
ax[3].plot(steps/1000, 'o-', label='ksteps'); ax[3].plot(reps, 's-', label='replicas')
ax[3].set_title('adaptive budget'); ax[3].legend()
for a in ax: a.set_xlabel('iteration')
plt.tight_layout(); plt.show()

### 5b. Behavioural check

Parameter recovery is necessary but not sufficient — does the *fitted* model
reproduce the target's behaviour, and does it reach the replica-noise ceiling?

Tagged with the best iteration so it can never collide with a previous fit's
cached ensemble.

In [ ]:
FIT_TAG = f'fitted_i{res.best_iter}'
P_fit = sim.run_ensemble(FIT_TAG, model=res.model, n_replicas=10,
                         save_spec=SaveSpec(contact_map=True, trajectory=False,
                                            monitor=False),
                         n_production=3_000_000, verbose=False)

iu = np.triu_indices(target.shape[0], 3)
r_fit = np.corrcoef(target[iu], P_fit[iu])[0,1]
print(f'fitted vs target correlation = {r_fit:.5f}')
print(f'replica-noise ceiling        = {ceiling:.5f}')
print('->', 'AT the ceiling: recovered as well as the data allows'
      if r_fit >= ceiling - 0.01 else 'below the ceiling: residual error remains')

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a,(Mp,t,cm) in zip(ax, [(target,'target (truth)','Reds'),
                            (P_fit,'fitted','Reds'),
                            (P_fit-target,'difference','RdBu_r')]):
    im = a.imshow(Mp, cmap=cm); a.set_title(t); plt.colorbar(im, ax=a)
plt.tight_layout(); plt.show()

## 6. Free C (blind reduced-rank)

Fit **both** C and Λ from a random start — no biological prior. This is where
gauge freedom bites: C and Λ are determined only up to C → MC, Λ → M⁻ᵀΛM⁻¹,
which leaves M unchanged. Judge recovery on M, never on raw C or Λ.

The Procrustes **rotation magnitude** is the identifiability diagnostic:
near-identity means the parameterization is pinned; large means a flat direction.

In [ ]:
RUN_FREE_C = False    # enable after k=1 passes

if RUN_FREE_C:
    free_init = Model.free(N_BEADS, k=1, seed=3, scale=0.5)
    free_init.names = names; free_init.c = -0.30
    res2 = O.Optimizer(sim, target, lr_lambda=0.05, lr_C=0.01,
                       budget=O.Budget(n_steps=50_000, n_replicas=4),
                       out_dir=f'{OUT}/fits', tag='freeC', verbose=True
                       ).fit(free_init, n_iter=80, patience=15)
    rep2 = O.recovery_report(true_model, res2.model)
    print(f"\nM relative error   : {rep2['M_relative_error']:.4f}")
    print(f"M scale ratio      : {rep2['M_scale_ratio']:.4f}")
    if 'rotation_magnitude' in rep2:
        print(f"rotation magnitude : {rep2['rotation_magnitude']:.4f}")
else:
    print('skipped — set RUN_FREE_C = True after k=1 passes')

## 7. What this establishes

If λ came back inside the flat region measured in 3b:
- the max-ent gradient and its projection onto (C, Λ) are correct,
- warm-starting and the adaptive budget work,
- **the optimizer can be trusted where the answer is unknown.**

The width of that flat region is the identifiability result: the precision with
which a contact map determines this parameter.

**Next:** the chr10 continuous fit — fitting λ for the real PC1 coordinate,
removing the inherited-gauge caveat from the completed `cont` arm, where the
scale was chosen by hand and overshot experiment (3.06 vs 2.76).